# Valuing European-Style Swaptions with Matlab Example

We value a European swaption using Black's model and try to replicate a ML example at https://fr.mathworks.com/help/fininst/swaptionbyblk.html

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from financepy.utils.date import Date
from financepy.utils.global_vars import ONE_MILLION
from financepy.utils.day_count import DayCountTypes
from financepy.utils.frequency import FrequencyTypes
from financepy.utils.calendar import CalendarTypes, BusDayAdjustTypes, DateGenRuleTypes

from financepy.market.curves.interpolator import InterpTypes
from financepy.market.curves.ibor_single_curve import IborSingleCurve
from financepy.market.curves.flat_discount_curve import FlatDiscountCurve
from financepy.market.curves.zero_rates_discount_curve import ZeroRatesDiscountCurve

from financepy.products.rates.ibor_deposit import IborDeposit
from financepy.products.rates.ibor_swap import IborSwap, SwapTypes
from financepy.products.rates.ibor_swaption import IborSwaption, BlackShifted, Black, SABR, SABRShifted, HWTree, BKTree

#############################################################
#  FINANCEPY Version 1.1.2 - Built on 25 Sep 2026 at 14:07  #
#  This software is distributed FREE AND WITHOUT WARRANTY   #
#  Report issues at https://github.com/domokane/FinancePy   #
#############################################################



In [3]:
value_dt = Date(1, 1, 2010)

In [4]:
libor_curve = FlatDiscountCurve(value_dt, 0.06,
                                  FrequencyTypes.CONTINUOUS,
                                  DayCountTypes.THIRTY_E_360)

### Defining the swaption

In [5]:
settle_dt = Date(1, 1, 2011)
exercise_date = Date(1, 1, 2016)
maturity_dt = Date(1, 1, 2019)
fixed_cpn = 0.062
fixed_freq_type = FrequencyTypes.SEMI_ANNUAL
fixed_dc_type = DayCountTypes.THIRTY_E_360
float_freq_type = FrequencyTypes.SEMI_ANNUAL
float_dc_type = DayCountTypes.THIRTY_E_360
notional = 100.0
swap_type = SwapTypes.PAY
cal_type = CalendarTypes.NONE
bd_type = BusDayAdjustTypes.NONE
dg_type = DateGenRuleTypes.BACKWARD

In [6]:
swaption = IborSwaption(settle_dt,
                            exercise_date,
                            maturity_dt,
                            swap_type,
                            fixed_cpn,
                            fixed_freq_type,
                            fixed_dc_type,
                            notional,
                            float_freq_type,
                            float_dc_type,
                            cal_type,
                            bd_type,
                            dg_type)

In [7]:
print(swaption)

OBJECT_TYPE: IborSwaption
SETTLEMENT DATE: 01-JAN-2011
EXERCISE DATE: 01-JAN-2016
SWAP FIXED LEG TYPE: SwapTypes.PAY
SWAP MATURITY DATE: 01-JAN-2019
SWAP NOTIONAL: 100.0
FIXED cpn: 6.2
FIXED FREQUENCY: FrequencyTypes.SEMI_ANNUAL
FIXED DAY COUNT: DayCountTypes.THIRTY_E_360
FLOAT FREQUENCY: FrequencyTypes.SEMI_ANNUAL
FLOAT DAY COUNT: DayCountTypes.THIRTY_E_360



## Valuation using Black's Model

In [8]:
model = Black(0.20)

In [9]:
swaption.value(value_dt, libor_curve, model)

np.float64(2.0715673101227794)

The MATLAB price is 2.071. 

In [10]:
print(swaption)

OBJECT_TYPE: IborSwaption
SETTLEMENT DATE: 01-JAN-2011
EXERCISE DATE: 01-JAN-2016
SWAP FIXED LEG TYPE: SwapTypes.PAY
SWAP MATURITY DATE: 01-JAN-2019
SWAP NOTIONAL: 100.0
FIXED cpn: 6.2
FIXED FREQUENCY: FrequencyTypes.SEMI_ANNUAL
FIXED DAY COUNT: DayCountTypes.THIRTY_E_360
FLOAT FREQUENCY: FrequencyTypes.SEMI_ANNUAL
FLOAT DAY COUNT: DayCountTypes.THIRTY_E_360
PV01: 1.8868795344638085
FWD SWAP RATE: 6.090906790703374
FWD DF TO EXPIRY: 0.697676326071031


We can see that the forward swap rate almost equals the fixed cpn. The underlying swap is close to being ATM forward.

In [11]:
swaption.print_swap_fixed_leg()

START DATE: 01-JAN-2016
MATURITY DATE: 01-JAN-2019
COUPON (%): 6.2
FREQUENCY: FrequencyTypes.SEMI_ANNUAL
DAY COUNT: DayCountTypes.THIRTY_E_360

PAYMENTS VALUATION:
PAY_NUM | PAY_dt      | NOTIONAL | RATE | PMNT | DF     | PV   | CUM_PV
--------+-------------+----------+------+------+--------+------+-------
      1 | 01-JUL-2016 |    100.0 |  6.2 |  3.1 | 0.6771 |  2.1 |    2.1
      2 | 01-JAN-2017 |    100.0 |  6.2 |  3.1 |  0.657 | 2.04 |   4.14
      3 | 01-JUL-2017 |    100.0 |  6.2 |  3.1 | 0.6376 | 1.98 |   6.11
      4 | 01-JAN-2018 |    100.0 |  6.2 |  3.1 | 0.6188 | 1.92 |   8.03
      5 | 01-JUL-2018 |    100.0 |  6.2 |  3.1 | 0.6005 | 1.86 |   9.89
      6 | 01-JAN-2019 |    100.0 |  6.2 |  3.1 | 0.5827 | 1.81 |   11.7


## Increasing Yield Curve

In [12]:
value_dt = Date(1, 1, 2010)

In [13]:
dates = [Date(1,1,2011), Date(1,1,2012), Date(1,1,2013),
         Date(1,1,2014), Date(1,1,2015)]
rates = [0.03, 0.034, 0.037, 0.039, 0.04]

freq_type = FrequencyTypes.CONTINUOUS
day_count_type = DayCountTypes.THIRTY_E_360

In [14]:
libor_curve = ZeroRatesDiscountCurve(value_dt, dates, rates, freq_type, day_count_type, InterpTypes.LINEAR_ZERO_RATES)

In [15]:
print(libor_curve)

OBJECT TYPE: ZeroRatesDiscountCurve
ZERO RATE FREQUENCY: FrequencyTypes.CONTINUOUS
DATES: ZERO RATES
01-JAN-2010:          nan
01-JAN-2011:   0.03000000
01-JAN-2012:   0.03400000
01-JAN-2013:   0.03700000
01-JAN-2014:   0.03900000
01-JAN-2015:   0.04000000
CURVE DC_TYPE: DayCountTypes.THIRTY_E_360

OBJECT_TYPE: DiscountCurve
VALUE DATE: 01JAN2010
    DATES      TIMES(YRS) DISC FACTORS
 01-JAN-2010:     0.000000  1.00000000
 01-JAN-2011:     1.000000  0.97044553
 01-JAN-2012:     2.000000  0.93426047
 01-JAN-2013:     3.000000  0.89493875
 01-JAN-2014:     4.000000  0.85555919
 01-JAN-2015:     5.000000  0.81873075
INTERPOLATION TYPE: LINEAR_ZERO_RATES
CURVE DAY COUNT TYPE: THIRTY_E_360



In [16]:
settle_dt = Date(1, 1, 2011)
exercise_date = Date(1, 1, 2012)
maturity_dt = Date(1, 1, 2017)
fixed_cpn = 0.03
fixed_freq_type = FrequencyTypes.SEMI_ANNUAL
fixed_dc_type = DayCountTypes.THIRTY_E_360
notional = 1000.0
swaptionType = SwapTypes.RECEIVE

In [17]:
swaption = IborSwaption(settle_dt,
                            exercise_date,
                            maturity_dt,
                            swaptionType,
                            fixed_cpn,
                            fixed_freq_type,
                            fixed_dc_type,
                            notional)

In [18]:
model = Black(0.21)

In [19]:
swaption.value(value_dt, libor_curve, model)

np.float64(0.577555894365057)

This differs from Matlab who find 0.5771.

In [20]:
print(swaption)

OBJECT_TYPE: IborSwaption
SETTLEMENT DATE: 01-JAN-2011
EXERCISE DATE: 01-JAN-2012
SWAP FIXED LEG TYPE: SwapTypes.RECEIVE
SWAP MATURITY DATE: 01-JAN-2017
SWAP NOTIONAL: 1000.0
FIXED cpn: 3.0
FIXED FREQUENCY: FrequencyTypes.SEMI_ANNUAL
FIXED DAY COUNT: DayCountTypes.THIRTY_E_360
FLOAT FREQUENCY: FrequencyTypes.QUARTERLY
FLOAT DAY COUNT: DayCountTypes.THIRTY_E_360
PV01: 4.158509036040892
FWD SWAP RATE: 4.293863542920194
FWD DF TO EXPIRY: 0.9342604735772135


Let's just check the swap rate

In [21]:
libor_curve.swap_rate(exercise_date,
                   maturity_dt,
                   FrequencyTypes.SEMI_ANNUAL,
                   DayCountTypes.THIRTY_E_360)

np.float64(0.042938635429201975)

As required, it's the same.

In [22]:
swaption.print_swap_fixed_leg()

START DATE: 01-JAN-2012
MATURITY DATE: 02-JAN-2017
COUPON (%): 3.0
FREQUENCY: FrequencyTypes.SEMI_ANNUAL
DAY COUNT: DayCountTypes.THIRTY_E_360

PAYMENTS VALUATION:
PAY_NUM | PAY_dt      | NOTIONAL | RATE | PMNT  | DF     | PV    | CUM_PV
--------+-------------+----------+------+-------+--------+-------+-------
      1 | 02-JUL-2012 |   1000.0 |  3.0 | 15.08 |  0.915 |  13.8 |   13.8
      2 | 01-JAN-2013 |   1000.0 |  3.0 | 14.92 | 0.8949 | 13.35 |  27.15
      3 | 01-JUL-2013 |   1000.0 |  3.0 |  15.0 | 0.8755 | 13.13 |  40.28
      4 | 01-JAN-2014 |   1000.0 |  3.0 |  15.0 | 0.8556 | 12.83 |  53.12
      5 | 01-JUL-2014 |   1000.0 |  3.0 |  15.0 | 0.8372 | 12.56 |  65.67
      6 | 01-JAN-2015 |   1000.0 |  3.0 |  15.0 | 0.8187 | 12.28 |  77.95
      7 | 01-JUL-2015 |   1000.0 |  3.0 |  15.0 | 0.8025 | 12.04 |  89.99
      8 | 01-JAN-2016 |   1000.0 |  3.0 |  15.0 | 0.7866 |  11.8 | 101.79
      9 | 01-JUL-2016 |   1000.0 |  3.0 |  15.0 | 0.7711 | 11.57 | 113.36
     10 | 02-JAN-2017 

Differences with Matlab are likely due to rate compounding or calculation of time used in volatility calculations.

Copyright (c) 2020 Dominic O'Kane